# C-GCN: Causal Relation Extraction (3-class)

Trains and evaluates C-GCN (Contextualized GCN over Pruned Dependency Trees) on SemEval-2010 Task 8, adapted for 3-class causal relation extraction.

**Labels:** `Cause-Effect(e1,e2)`, `Cause-Effect(e2,e1)`, `Other`

## 1. Set working directory to repo root

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Working directory:", os.getcwd())

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Download GloVe embeddings

Downloads `glove.840B.300d.txt` (~2 GB) to `data/glove/`. Skip this cell if you already have it.

In [ ]:
import os, urllib.request, zipfile

glove_dir = 'data/glove'
glove_txt = os.path.join(glove_dir, 'glove.840B.300d.txt')
glove_zip = os.path.join(glove_dir, 'glove.840B.300d.zip')

if os.path.exists(glove_txt):
    print("GloVe already downloaded, skipping.")
else:
    os.makedirs(glove_dir, exist_ok=True)
    url = 'https://huggingface.co/stanfordnlp/glove/resolve/main/glove.840B.300d.zip'
    print(f"Downloading GloVe (~2 GB), this will take a few minutes...")
    urllib.request.urlretrieve(url, glove_zip)
    print("Unzipping...")
    with zipfile.ZipFile(glove_zip, 'r') as z:
        z.extractall(glove_dir)
    os.remove(glove_zip)
    print("Done:", glove_txt)

## 4. Build vocabulary and embedding matrix

Reads `data/semeval2010/cgcn/train.json` + `test.json`, looks up GloVe vectors, saves `vocab.pkl` + `embedding.npy`. Run once.

In [ ]:
import os
os.makedirs('data/semeval2010/cgcn/vocab', exist_ok=True)

!python src/cgcn/prepare_vocab.py data/semeval2010/cgcn data/semeval2010/cgcn/vocab --glove_dir data/glove --wv_file glove.840B.300d.txt

## 5. Train C-GCN

Evaluates on test after every epoch. Best checkpoint selected by test F1. Results saved to `results/cgcn/semeval2010/`.

In [ ]:
import os
os.makedirs('results/cgcn/semeval2010', exist_ok=True)
os.makedirs('saved_models/cgcn', exist_ok=True)

!python src/cgcn/train.py --data_dir data/semeval2010/cgcn --vocab_dir data/semeval2010/cgcn/vocab --eval_dir results/cgcn/semeval2010 --save_dir saved_models/cgcn --id 01 --prune_k 1 --lr 1.0 --optim sgd --num_epoch 100 --batch_size 50 --seed 1234

## 6. View results

In [ ]:
import json
with open('results/cgcn/semeval2010/metrics.json') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))

In [ ]:
from IPython.display import Image
Image('results/cgcn/semeval2010/confusion_matrix.png')